# BirdCLEF 2026 — Mel-Only Training v25 (ResNet18 + EfficientNet-B0)

Standalone mel model training — no Perch / ONNX / embeddings needed.
Runs on any GPU (Kaggle T4, GCP, Colab, etc.).

### Outputs
- `resnet18_v25_fold0.pt` … `resnet18_v25_fold4.pt`
- `efficientnet_b0_v25_fold0.pt` … `efficientnet_b0_v25_fold4.pt`

### Required inputs
- `birdclef-2026` competition dataset (train_audio, taxonomy.csv, train.csv)

### Design
- Identical architecture/hyperparams to mel branch in `birdclef2026-train-v25-gru-mel.ipynb`
- Pre-computes 16kHz waveform cache once → fast `np.load` every step
- SpecAugment (frequency + time masking)
- 5-fold stratified cross-validation, early stopping, cosine LR with warmup

In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, ast, copy, random, multiprocessing as _mp
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import timm

CFG = dict(
    seed        = 42,
    folds       = 5,
    device      = 'cuda' if torch.cuda.is_available() else 'cpu',

    # Mel spectrogram
    mel_sr      = 16000,
    mel_seconds = 5,
    n_mels      = 64,
    n_fft       = 1024,
    hop_length  = 320,
    fmin        = 60,
    fmax        = 8000,

    # Training
    mel_batch         = 32,
    mel_lr            = 3e-4,
    mel_epochs        = 25,
    mel_patience      = 7,
    warmup_epochs     = 4,
    num_workers       = 4,
    secondary_label_weight = 0.3,
    checkpoint_tag    = 'v25',
)
CFG['mel_target'] = CFG['mel_sr'] * CFG['mel_seconds']  # 80,000

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device   = torch.device(CFG['device'])
_use_amp = (device.type == 'cuda')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Device   : {device}  AMP={_use_amp}')
print(f'Folds    : {CFG["folds"]}  Epochs: {CFG["mel_epochs"]}  Batch: {CFG["mel_batch"]}')

In [ ]:
# === CELL 2: PATHS & SPECIES ===
# ---- Configure for your environment ----
_env = 'kaggle'   # 'kaggle' | 'gcp' | 'colab'

if _env == 'kaggle':
    COMP_DIR       = '/kaggle/input/birdclef-2026'
    OUT_DIR        = '/kaggle/working'
    MEL_CACHE_DIR  = Path('/kaggle/working/mel_cache')
elif _env == 'gcp':
    COMP_DIR       = '/home/user/birdclef-2026'
    OUT_DIR        = '/home/user/output'
    MEL_CACHE_DIR  = Path('/home/user/mel_cache')
elif _env == 'colab':
    COMP_DIR       = '/content/birdclef-2026'
    OUT_DIR        = '/content/output'
    MEL_CACHE_DIR  = Path('/content/mel_cache')
else:
    raise ValueError(f'Unknown env: {_env}')

TRAIN_META_CSV  = f'{COMP_DIR}/train.csv'
TAXONOMY_CSV    = f'{COMP_DIR}/taxonomy.csv'
TRAIN_AUDIO_DIR = f'{COMP_DIR}/train_audio'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
species_set = set(species)
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)
df          = pd.read_csv(TRAIN_META_CSV)

with open(f'{OUT_DIR}/species_v25.json', 'w') as f:
    json.dump(species, f)

print(f'n_classes        : {n_classes}')
print(f'Train clips      : {len(df)}')
print(f'Train audio dir  : {TRAIN_AUDIO_DIR}')
print(f'Mel cache dir    : {MEL_CACHE_DIR}')
print(f'Output dir       : {OUT_DIR}')

In [ ]:
# === CELL 3: LABEL & SPECTROGRAM HELPERS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ('', '[]'): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype='float32')
    if str(primary_id) in sp_idx:
        y[sp_idx[str(primary_id)]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in sp_idx:
            y[sp_idx[sid]] = CFG['secondary_label_weight']
    return y

# Mel filter bank
_mel_filter = librosa.filters.mel(
    sr=CFG['mel_sr'], n_fft=CFG['n_fft'],
    n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
)

def logmel_from_wave(wave_16k: np.ndarray) -> np.ndarray:
    tgt = CFG['mel_target']
    if len(wave_16k) < tgt:
        wave_16k = np.pad(wave_16k, (0, tgt - len(wave_16k)))
    elif len(wave_16k) > tgt:
        start = random.randint(0, len(wave_16k) - tgt)
        wave_16k = wave_16k[start:start + tgt]
    stft   = librosa.stft(wave_16k, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'],
                          window='hann', center=True)
    power  = np.abs(stft) ** 2
    mel    = _mel_filter @ power
    logmel = np.log1p(mel).astype(np.float32)
    return logmel

print('\u2705 Helpers defined')

In [ ]:
# === CELL 4: MODEL DEFINITION ===
class BirdCLEFModel(nn.Module):
    """ResNet18 or EfficientNet-B0 log-mel classifier (identical to v25 joint notebook)."""
    def __init__(self, arch: str, n_classes: int, pretrained: bool = True):
        super().__init__()
        if arch == 'resnet18':
            base    = timm.create_model('resnet18', pretrained=pretrained, in_chans=1)
            n_feats = base.fc.in_features
            base.fc = nn.Identity()
        elif arch == 'efficientnet_b0':
            base            = timm.create_model('efficientnet_b0', pretrained=pretrained, in_chans=1)
            n_feats         = base.classifier.in_features
            base.classifier = nn.Identity()
        else:
            raise ValueError(f'Unknown arch: {arch}')
        self.backbone = base
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(n_feats, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x):  # x: (B, 1, n_mels, T_frames)
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = self.pool(feats).flatten(1)
        return self.head(feats)

_r = BirdCLEFModel('resnet18', 10)
print(f'ResNet18       : {sum(p.numel() for p in _r.parameters())/1e6:.2f}M params')
del _r
_e = BirdCLEFModel('efficientnet_b0', 10)
print(f'EfficientNet-B0: {sum(p.numel() for p in _e.parameters())/1e6:.2f}M params')
del _e
print('\u2705 BirdCLEFModel defined')

In [ ]:
# === CELL 5: MEL DATASET ===
class MelFocalDataset(Dataset):
    """
    Loads log-mel spectrogram from cache (fast np.load) or computes from raw audio.
    Cache stores pre-computed mel specs (~2 GB total vs ~80+ GB for full waveforms).
    SpecAugment (freq + time masking) is still applied during training.
    """
    def __init__(self, frame: pd.DataFrame, audio_root: str, train: bool,
                 cache_dir: Path = None):
        self.df         = frame.reset_index(drop=True)
        self.audio_root = Path(audio_root)
        self.train      = train
        self.cache_dir  = Path(cache_dir) if cache_dir is not None else None

    def __len__(self): return len(self.df)

    @staticmethod
    def cache_key(filename: str) -> str:
        return str(filename).replace('/', '_').rsplit('.', 1)[0]

    def _load_wave(self, filepath: str) -> np.ndarray:
        try:
            y, sr = sf.read(filepath, always_2d=False)
            if y.ndim == 2: y = y.mean(1)
            if sr != CFG['mel_sr']:
                y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['mel_sr'])
            return y.astype(np.float32)
        except Exception:
            return np.zeros(CFG['mel_target'], dtype=np.float32)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        # Try mel cache first (pre-computed, center-cropped)
        lm = None
        if self.cache_dir is not None:
            cached = self.cache_dir / (self.cache_key(str(r['filename'])) + '.npy')
            if cached.exists():
                lm = np.load(str(cached))  # (n_mels, T_frames)

        if lm is None:
            # Fallback: load raw audio, compute mel with random crop
            wave = self._load_wave(str(self.audio_root / str(r['filename'])))
            lm   = logmel_from_wave(wave)

        # SpecAugment (applied even on cached mels — provides per-epoch variation)
        if self.train:
            if random.random() < 0.5:
                f0 = random.randint(0, CFG['n_mels'] - 1)
                fw = random.randint(1, min(10, CFG['n_mels'] - f0))
                lm[f0:f0+fw, :] = 0.0
            if random.random() < 0.5:
                T  = lm.shape[1]
                t0 = random.randint(0, max(0, T - 1))
                tw = random.randint(1, min(20, T - t0))
                lm[:, t0:t0+tw] = 0.0

        lm = (lm - lm.mean()) / (lm.std() + 1e-6)
        x  = torch.from_numpy(lm).float().unsqueeze(0)  # (1, n_mels, T_frames)
        y  = torch.from_numpy(
            row_to_multihot(r['primary_label'], r.get('secondary_labels', '[]'))
        ).float()
        return x, y


# Prepare mel dataframe
mel_df = df.copy()
if 'secondary_labels' not in mel_df.columns:
    mel_df['secondary_labels'] = '[]'
else:
    mel_df['secondary_labels'] = mel_df['secondary_labels'].fillna('[]')
mel_df = mel_df[mel_df['primary_label'].isin(species_set)].reset_index(drop=True)
print(f'Mel training clips: {len(mel_df)}')
print('✅ MelFocalDataset defined (mel-spec cache)')

In [ ]:
# === CELL 6: PRE-COMPUTE MEL SPECTROGRAM CACHE (run once, ~10-20 min) ===
# Stores log-mel specs instead of full waveforms.
# Size: ~2.3 GB total  vs  ~80+ GB for full waveforms — fits easily on Kaggle 20 GB.
# Crop position is random at cache time so calls at any position are captured.
# SpecAugment is still applied per-epoch from the cached mels during training.
MEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_unique_files = mel_df['filename'].drop_duplicates().tolist()

# All mel params passed explicitly — no global captures, safe for multiprocessing
def _precompute_one(args):
    filename, audio_root, cache_dir, mel_sr, mel_target, n_fft, hop_length, n_mels, fmin, fmax = args
    key     = str(filename).replace('/', '_').rsplit('.', 1)[0]
    out_npy = Path(cache_dir) / f'{key}.npy'
    if out_npy.exists():
        return 'skip'
    import soundfile as _sf
    import librosa as _lb
    import numpy as _np
    import random as _rng
    filepath = Path(audio_root) / str(filename)
    try:
        y, sr = _sf.read(str(filepath), always_2d=False)
        if y.ndim == 2: y = y.mean(1)
        if sr != mel_sr:
            y = _lb.resample(y.astype(_np.float32), orig_sr=sr, target_sr=mel_sr)
        # Random crop — bird calls can be anywhere in the recording
        if len(y) < mel_target:
            y = _np.pad(y, (0, mel_target - len(y)))
        else:
            start = _rng.randint(0, len(y) - mel_target)
            y = y[start:start + mel_target]
        # Compute log-mel spectrogram
        mel_fb = _lb.filters.mel(sr=mel_sr, n_fft=n_fft, n_mels=n_mels, fmin=fmin, fmax=fmax)
        stft   = _lb.stft(y, n_fft=n_fft, hop_length=hop_length, window='hann', center=True)
        power  = _np.abs(stft) ** 2
        mel    = mel_fb @ power
        logmel = _np.log1p(mel).astype(_np.float32)  # shape: (n_mels, T_frames)
        _np.save(str(out_npy), logmel)
        return 'ok'
    except Exception as e:
        return f'err:{e}'

_already = sum(
    1 for fn in _unique_files
    if (MEL_CACHE_DIR / (MelFocalDataset.cache_key(fn) + '.npy')).exists()
)
_to_do   = len(_unique_files) - _already
_t_frames = CFG['mel_target'] // CFG['hop_length'] + 1
_est_gb   = len(_unique_files) * CFG['n_mels'] * _t_frames * 4 / 1e9
print(f'Cache status : {_already}/{len(_unique_files)} done, {_to_do} remaining')
print(f'Est. disk    : ~{_est_gb:.1f} GB total  (mel specs, not waveforms)')

if _to_do > 0:
    _args_list = [
        (fn, TRAIN_AUDIO_DIR, str(MEL_CACHE_DIR),
         CFG['mel_sr'], CFG['mel_target'], CFG['n_fft'], CFG['hop_length'],
         CFG['n_mels'], CFG['fmin'], CFG['fmax'])
        for fn in _unique_files
        if not (MEL_CACHE_DIR / (MelFocalDataset.cache_key(fn) + '.npy')).exists()
    ]
    _n_workers = min(os.cpu_count() or 4, 8)
    with _mp.Pool(_n_workers) as pool:
        results = list(tqdm(
            pool.imap(_precompute_one, _args_list, chunksize=64),
            total=len(_args_list), desc='Caching mel specs'
        ))
    _errors = [r for r in results if r.startswith('err')]
    if _errors:
        print(f'  ⚠️  {len(_errors)} errors. First 5: {_errors[:5]}')

print(f'✅ Cache complete: {len(list(MEL_CACHE_DIR.glob("*.npy")))} files in {MEL_CACHE_DIR}')

In [ ]:
# === CELL 7: TRAIN MEL FOLDS (ResNet18 + EfficientNet-B0) ===
print('=' * 65)
print(f'v25 Mel Training   {CFG["folds"]} folds  (ResNet18 + EfficientNet-B0)')
print(f'  Mel cache: {MEL_CACHE_DIR}')
print('=' * 65)

skf = StratifiedKFold(n_splits=CFG['folds'], shuffle=True, random_state=CFG['seed'])
mel_fold_scores = {arch: [] for arch in ['resnet18', 'efficientnet_b0']}

_strat_labels = mel_df['primary_label'].map(
    lambda x: x if mel_df['primary_label'].value_counts().get(x, 0) >= CFG['folds'] else '__rare__'
)

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(mel_df, _strat_labels)):
    print(f'\nFold {fold_idx + 1}/{CFG["folds"]}')

    mel_tr = mel_df.iloc[tr_idx].reset_index(drop=True)
    mel_va = mel_df.iloc[va_idx].reset_index(drop=True)

    tr_ds = MelFocalDataset(mel_tr, TRAIN_AUDIO_DIR, train=True,  cache_dir=MEL_CACHE_DIR)
    va_ds = MelFocalDataset(mel_va, TRAIN_AUDIO_DIR, train=False, cache_dir=MEL_CACHE_DIR)
    tr_dl = DataLoader(tr_ds, batch_size=CFG['mel_batch'], shuffle=True,
                       num_workers=CFG['num_workers'], drop_last=True,  pin_memory=_use_amp)
    va_dl = DataLoader(va_ds, batch_size=CFG['mel_batch'], shuffle=False,
                       num_workers=CFG['num_workers'], drop_last=False, pin_memory=_use_amp)

    for arch in ['resnet18', 'efficientnet_b0']:
        print(f'  Training {arch}...')
        model     = BirdCLEFModel(arch, n_classes, pretrained=True).to(device)
        optimizer = AdamW(model.parameters(), lr=CFG['mel_lr'], weight_decay=1e-4)
        scaler    = GradScaler(enabled=_use_amp)
        warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                             total_iters=CFG['warmup_epochs'])
        cosine    = CosineAnnealingLR(optimizer,
                                      T_max=max(1, CFG['mel_epochs'] - CFG['warmup_epochs']),
                                      eta_min=1e-6)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                                 milestones=[CFG['warmup_epochs']])

        best_auc    = -1.0
        patience_ct = 0
        best_state  = None

        for epoch in range(CFG['mel_epochs']):
            model.train()
            train_loss = 0.0
            for xb, yb in tr_dl:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    loss = F.binary_cross_entropy_with_logits(model(xb), yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
            train_loss /= max(len(tr_dl), 1)
            scheduler.step()

            model.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for xv, yv in va_dl:
                    xv = xv.to(device)
                    with autocast(enabled=_use_amp):
                        preds = torch.sigmoid(model(xv).float()).cpu().numpy()
                    val_preds.append(preds)
                    val_targets.append(yv.numpy())

            fp     = np.vstack(val_preds)
            ft     = np.vstack(val_targets)
            ft_bin = (ft >= 0.5).astype(np.float32)
            auc_ep = [
                roc_auc_score(ft_bin[:, j], fp[:, j])
                for j in range(n_classes)
                if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
            ]
            val_auc = np.mean(auc_ep) if auc_ep else 0.0

            if val_auc > best_auc:
                best_auc    = val_auc
                patience_ct = 0
                best_state  = copy.deepcopy(model.state_dict())
            else:
                patience_ct += 1

            if (epoch + 1) % 5 == 0 or patience_ct == 0:
                print(f'    Ep {epoch+1:3d}: train={train_loss:.4f}  auc={val_auc:.4f}')

            if patience_ct >= CFG['mel_patience']:
                print(f'    Early stop @ epoch {epoch+1}')
                break

        if best_state:
            model.load_state_dict(best_state)
        ckpt = f'{OUT_DIR}/{arch}_v25_fold{fold_idx}.pt'
        torch.save(model.state_dict(), ckpt)
        mel_fold_scores[arch].append(best_auc)
        print(f'    ✅ {arch} fold {fold_idx+1} AUC: {best_auc:.4f}  → {ckpt}')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

for arch, scores in mel_fold_scores.items():
    if scores:
        print(f'\n✅ {arch} Mean OOF AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}')
        print(f'   Fold AUCs: {[f"{a:.4f}" for a in scores]}')

In [ ]:
# === CELL 8: SUMMARY ===
saved = sorted(Path(OUT_DIR).glob('*_v25_fold*.pt'))
print('Checkpoints saved:')
for f in saved:
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

print()
for arch, scores in mel_fold_scores.items():
    if scores:
        print(f'{arch:20s} mean AUC: {np.mean(scores):.4f}')

print()
print('Next steps (Kaggle):')
print('  Output tab → New Dataset → name: birdclef-2026-weights-v25')
print('  Add alongside perch_gru_v25_fold*.pt from GRU training run')
print('  Inference CFG: gru_weight=0.6, mel_weight=0.4')